# BLIP Evaluation Script
This notebook contains the code from the provided `blip_eval.py` script.

In [1]:
# cd /PATH/TO/GR_MG
EPOCHS=(47)
CKPT_DIR="/tmp2/danzel/GR-MG/checkpoints/policy"
# export SD_CKPT="/PATH_TO_GOAL_GEN_MODEL_CKPT/epoch=49-step=51450.ckpt"
SD_CKPT="/tmp2/danzel/GR-MG/checkpoints/goal/goal_gen.ckpt"
MESA_GL_VERSION_OVERRIDE=3.3

# sudo chmod 777 -R ${CKPT_DIR}

COUNTER=1

In [18]:

# MIT License
# Copyright (c) 2021 Oier Mees
# Copyright (c) 2024 Bytedance Ltd. and/or its affiliates

import argparse
import json
import logging
import os
from pathlib import Path
import sys
import time
import re
import copy
from copy import deepcopy
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig, BlipProcessor, BlipForQuestionAnswering

# cuda device


# os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Restrict to GPU 1
torch.cuda.set_device(1)  # Set default GPU to GPU 1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # cuda:0 corresponds to GPU 1
print(f"Current GPU ID: {torch.cuda.current_device()}")

print(os.getcwd())

# Add the project root to the Python path
project_root = '/tmp2/danzel/GR-MG'  # Replace with the actual path to your project root
sys.path.append(project_root)

# Verify the updated path
print(sys.path)

from calvin_agent.evaluation.multistep_sequences import get_sequences
from calvin_agent.evaluation.utils import (
    count_success,
    get_env_state_for_initial_condition
)
import hydra
import numpy as np
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
from termcolor import colored
import torch
from tqdm.auto import tqdm
from utils.utils import print_and_save
from wrapper.model_wrapper import CustomModel
from goal_gen.evaluate import IP2PEvaluation

logger = logging.getLogger(__name__)
EP_LEN = 360
NUM_SEQUENCES = 1000
SAVE_DIR = None
FAIL_COUNTER=0

processor_blip2 = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model_blip2 = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", quantization_config=quantization_config, device_map={"": 0}, torch_dtype=torch.float16
)  # doctest: +IGNORE_RESULT

#VQA
processor_vqa = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model_vqa = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)

Current GPU ID: 5
/tmp2/danzel/GR-MG/evaluate
['/home/master/13/danzel/anaconda3/envs/gr-mg/lib/python39.zip', '/home/master/13/danzel/anaconda3/envs/gr-mg/lib/python3.9', '/home/master/13/danzel/anaconda3/envs/gr-mg/lib/python3.9/lib-dynload', '', '/home/master/13/danzel/anaconda3/envs/gr-mg/lib/python3.9/site-packages', '/tmp2/danzel/GR-MG/calvin/calvin_env/tacto', '/tmp2/danzel/GR-MG/calvin/calvin_models', '/tmp2/danzel/GR-MG', '/tmp/tmpw8q9wo9p', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG', '/tmp2/danzel/GR-MG']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 246.00 MiB. GPU 

In [ ]:

def make_env(dataset_path, observation_space, device_id):
    val_folder = Path(dataset_path) / "validation"
    from wrapper.calvin_env_wrapper_raw import CalvinEnvWrapperRaw
    device = torch.device('cuda', device_id)
    env = CalvinEnvWrapperRaw(val_folder, observation_space, device)
    return env


In [ ]:

def blip_prompt(img, lang_annotation, img_path):
    print(lang_annotation)
    print(img_path)
    raw_image = Image.open(img_path).convert("RGB")
    print(raw_image.size)
    question = "What direction should the white robot arm go next?"
    
    
    # Generate answer with blip2
    inputs = processor_blip2(images=raw_image, text=question, return_tensors="pt").to(device, torch.float16)
    generated_ids = model_blip2.generate(**inputs)
    answer = processor_blip2.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    #Gnerate answer with VQA
    inputs = processor_vqa(raw_image, question, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
    outputs = model_vqa(**inputs)
    answer = processor_vqa.post_process_qa_predictions(inputs, question, outputs, answer, version_2_with_negative=False)
    
    print(f"Question: {question}\nAnswer: {answer}")
    
    return answer


In [ ]:

def rollout(env, model, task_oracle, subtask, val_annotations, subtask_i, sequence_i, ip2p_model):
    obs = env.get_obs()
    lang_annotation = val_annotations[subtask][0]
    model.reset()
    start_info = env.get_info()
    debug_image = []
    progress = 0
    for i in range(EP_LEN):
        if i % 20 == 0:
            static_rgb = obs['rgb_obs']['rgb_static']
            hand_rgb = obs['rgb_obs']['rgb_gripper']
            text_prompt = blip_prompt(static_rgb, lang_annotation)
            goal_image = ip2p_model.inference([static_rgb], [lang_annotation + f". {progress}% completed."])
            debug_image.append([static_rgb, goal_image[0], hand_rgb])
        action, progress = model.step(obs, deepcopy(goal_image), [lang_annotation])
        obs, _, _, current_info = env.step(action)
        if len(task_oracle.get_task_info_for_set(start_info, current_info, {subtask})) > 0:
            return True
    return False


In [ ]:

def evaluate_sequence(env, model, task_checker, initial_state, eval_sequence, val_annotations, sequence_i, ip2p_model):
    robot_obs, scene_obs = get_env_state_for_initial_condition(initial_state)
    env.reset(robot_obs=robot_obs, scene_obs=scene_obs)
    success_counter = 0
    for subtask_i, subtask in enumerate(eval_sequence):
        success = rollout(env, model, task_checker, subtask, val_annotations, subtask_i, sequence_i, ip2p_model)
        if success:
            success_counter += 1
        else:
            return success_counter
    return success_counter


In [ ]:

def evaluate_policy(model, env, eval_sr_path, eval_result_path, ip2p_model):
    conf_dir = Path("./calvin/calvin_models/conf")
    task_cfg = OmegaConf.load(conf_dir / "callbacks/rollout/tasks/new_playtable_tasks.yaml")
    task_oracle = hydra.utils.instantiate(task_cfg)
    val_annotations = OmegaConf.load(conf_dir / "annotations/new_playtable_validation.yaml")
    eval_sequences = get_sequences(NUM_SEQUENCES)
    results = []
    sequence_i = 0
    for index,(initial_state, eval_sequence) in enumerate(eval_sequences):
        result = evaluate_sequence(
            env, model, task_oracle, initial_state, eval_sequence, val_annotations, sequence_i, ip2p_model)
        results.append(result)
        success_list = count_success(results)
        with open(eval_sr_path, 'a') as f:
            line = f"{sequence_i}/{NUM_SEQUENCES}: "
            for sr in success_list:
                line += f"{sr:.3f} | "
            sequence_i += 1
            line += "\n"
            f.write(line)

        if index % 100 == 0 and index != 0:  # save every 100 sequences
            print_and_save(results, eval_sequences[:index+1], eval_result_path[:-5] + f"_{index+1}.json", None)
    print_and_save(results, eval_sequences, eval_result_path, None)
    return results


In [ ]:
seed_everything(0, workers=True)
parser = argparse.ArgumentParser(description="Evaluate a trained model on multistep sequences with language goals.")
parser.add_argument("--dataset_path", type=str, help="Path to the dataset root directory.", default="/tmp2/danzel/GR-MG/data")
parser.add_argument("--config_path", type=str, help="Path to the policy config file.", default="/tmp2/danzel/GR-MG/configs/policy_config.json")
parser.add_argument("--ckpt_dir", type=str, help="Path to the policy checkpoint directory.",default="/tmp2/danzel/GR-MG/checkpoints/policy")
parser.add_argument("--epoch", type=int, help="Epoch index for evaluation.",default=47)
parser.add_argument("--device_id", type=int, help="CUDA device ID.",default=0)
parser.add_argument("--ip2p_ckpt_path", type=str, help="Path to the IP2P model checkpoint.",default="/tmp2/danzel/GR-MG/checkpoints/goal/goal_gen.ckpt")
args = parser.parse_args()
config_path = args.config_path
ckpt_dir = args.ckpt_dir
epoch = args.epoch
device_id = args.device_id
ip2p_ckpt_path=args.ip2p_ckpt_path

ip2p_model = IP2PEvaluation(args.ip2p_ckpt_path)

In [ ]:
# Load config file
with open(config_path, 'r') as f:
    configs = json.load(f)
            
# Get checkpoint path
ckpt_path = None
ckpt_files = os.listdir(ckpt_dir)
for ckpt_file in ckpt_files:
    match = re.search(r'epoch=(\d+)', ckpt_file)
    if match:
        temp_epoch = int(match.group(1))
        if temp_epoch == epoch:
            ckpt_path = os.path.join(ckpt_dir, ckpt_file)
            break

device = torch.device('cuda', device_id)
model = CustomModel(
    ckpt_path=ckpt_path,
    configs=configs,
    device=device)
observation_space = {
    'rgb_obs': ['rgb_static', 'rgb_gripper'], 
    'depth_obs': [], 
    'state_obs': ['robot_obs'], 
    'actions': ['rel_actions'], 
    'language': ['language']} 
env = make_env(args.dataset_path, observation_space, device_id) 
print("Start evaluating policy...")
# Success rate and result files
flag="opensourcesd"
sub_dir=f"{flag}_{epoch}_epoch"
# set a global variable
global SAVE_DIR
SAVE_DIR=os.path.join(ckpt_dir,sub_dir)
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
sr_path = os.path.join(SAVE_DIR, f"success_rate.txt")
result_path = os.path.join(SAVE_DIR, f"results.json")


In [ ]:
evaluate_policy(model, env, os.path.join(args.ckpt_dir, "success_rate.txt"), os.path.join(args.ckpt_dir, "results.json"), ip2p_model)